---
title: Reciprocity Expansion Notebook
authors: [gvarnavides]
date: 2026-05-28
---

In [1]:
%matplotlib widget

import abtem
import numpy as np
import quantem as em
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable

from matplotlib.gridspec import GridSpec
import ipywidgets

plt.rcParams['text.color']='white'
plt.rcParams['xtick.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.labelcolor'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['axes.edgecolor'] = 'white'
plt.rcParams.update({
    "text.usetex": True,
    "text.latex.preamble": r"\usepackage{amsmath}"
})

abtem.config.set({"dask.lazy":False});


### Reciprocity Expansion

In [2]:
def array_to_scaled_rgba(array,vmin=0.02,vmax=0.98):
    if np.iscomplexobj(array):
        scaled_amplitude = np.abs(array)
        scaled_angle = np.angle(array)
    else:
        scaled_amplitude = array
        scaled_angle = None

    if scaled_amplitude.std() > 1e-12:
        vmin, vmax = np.quantile(scaled_amplitude,(vmin,vmax))

        scaled_amplitude = (scaled_amplitude.clip(vmin,vmax) - vmin) / (vmax-vmin)
    else:
        scaled_amplitude = np.ones_like(scaled_amplitude)

    rgba = em.visualization.visualization_utils.array_to_rgba(
        scaled_amplitude,
        scaled_angle
    )

    return rgba

In [3]:
# parameters
gpts = np.array([128,128])
sampling = np.array([0.125,0.125])
extent = gpts*sampling
energy = 300e3
semiangle_cutoff = 20
defocus = 100

In [4]:
arrays_mutated = [
    None, # s_matrix
    None, # spiral_ordering
    None, # indices_i
    None, # indices_j
    None, # pos_px
    None, # position_coefs
    None, # defocus
    None, # ctf_coefs
]

In [5]:
interpolation_factor = 1

In [6]:
dummy_s_matrix = abtem.SMatrix(
    potential=None,
    sampling=sampling,
    gpts=gpts,
    energy=energy,
    semiangle_cutoff=semiangle_cutoff,
    downsample=None,
    interpolation=interpolation_factor,
).build(
)

arrays_mutated[0] = dummy_s_matrix

In [7]:
def return_ordering_indices(s_matrix):
    wave_vectors = s_matrix.wave_vectors
    spiral_ordering = np.argsort(np.sum(wave_vectors**2,1))
    
    indices_i, indices_j = np.mod(
        (wave_vectors * s_matrix.extent).astype("int"),
        s_matrix.gpts
    ).T

    return spiral_ordering, indices_i, indices_j

arrays_mutated[1:4] = return_ordering_indices(arrays_mutated[0])

In [8]:
# inputs b

defocus = 0
pos_px = gpts/2

In [9]:
def return_position_coefs(s_matrix, pos_px):
    pos = (pos_px-gpts/2) * np.array(s_matrix.sampling)
    position_coefs = s_matrix._calculate_positions_coefficients(
        abtem.scan.CustomScan(pos),
    )[0]
    return position_coefs

def return_ctf_coefs(s_matrix, defocus):
    ctf_coefs = s_matrix._calculate_ctf_coefficients(
        ctf=abtem.CTF(
            defocus=defocus,
            semiangle_cutoff=np.inf,
            energy=energy
        )
    )
    return ctf_coefs

arrays_mutated[4] = pos_px
arrays_mutated[5] = return_position_coefs(arrays_mutated[0],arrays_mutated[4])

arrays_mutated[6] = defocus
arrays_mutated[7] = return_ctf_coefs(arrays_mutated[0],arrays_mutated[6])

In [10]:
num_planewaves = 256

In [11]:
def return_selected_inds(
    num_planewaves,
    spiral_ordering,
    indices_i,
    indices_j,
):
    inds = spiral_ordering[:num_planewaves]
    inds_i = indices_i[inds]
    inds_j = indices_j[inds]
    return inds, inds_i, inds_j 

def return_arrays(
    s_matrix,
    inds,
    inds_i,
    inds_j,
    position_coefs,
    ctf_coefs,
):
    coefs = ctf_coefs[inds] * position_coefs[inds]
    beams = np.zeros(s_matrix.gpts,dtype=np.complex64)
    beams[inds_i,inds_j]= coefs
    scaled_beams = s_matrix.array[inds] * coefs[:,None,None]
    probe = scaled_beams.sum(0)
    return beams, scaled_beams, probe

inds, inds_i, inds_j = return_selected_inds(
    num_planewaves,
    arrays_mutated[1],
    arrays_mutated[2],
    arrays_mutated[3]
)

beams, scaled_beams, probe = return_arrays(
    arrays_mutated[0],
    inds,
    inds_i,
    inds_j,
    arrays_mutated[5],
    arrays_mutated[7]
)

In [12]:
width = 620
aspect_ratio = 0.35
height = int(width * aspect_ratio)
dpi = 72
with plt.ioff():
    prism_fig = plt.figure(figsize=(width/dpi,height/dpi),dpi=dpi)
    
spec = GridSpec(2,6,figure=prism_fig)
ax1 = prism_fig.add_subplot(spec[:,:2])
ax2a = prism_fig.add_subplot(spec[0,2])
ax2b = prism_fig.add_subplot(spec[0,3])
ax2c = prism_fig.add_subplot(spec[1,2])
ax2d = prism_fig.add_subplot(spec[1,3])
ax3 = prism_fig.add_subplot(spec[:,4:])

beams_rgb = array_to_scaled_rgba(np.fft.fftshift(beams),vmin=0,vmax=1)
im_beams = ax1.imshow(beams_rgb)
ax1.set_title("Fourier-space beam coefficients")

plane_tgb_tl = array_to_scaled_rgba(scaled_beams[np.maximum(-num_planewaves,-4)],vmin=0,vmax=1)
im_plane_tl = ax2a.imshow(plane_tgb_tl)

if num_planewaves > 1:
    plane_tgb_tr = array_to_scaled_rgba(scaled_beams[np.maximum(-num_planewaves+1,-3)],vmin=0,vmax=1)
    im_plane_tr = ax2b.imshow(plane_tgb_tr)
else:
    im_plane_tr = ax2b.imshow(np.ones_like(plane_tgb_tl))
    ax2b.set_visible(False)

if num_planewaves > 2:
    plane_tgb_bl = array_to_scaled_rgba(scaled_beams[np.maximum(-num_planewaves+2,-2)],vmin=0,vmax=1)
    im_plane_bl = ax2c.imshow(plane_tgb_bl)
else:
    im_plane_bl = ax2c.imshow(np.ones_like(plane_tgb_tl))
    ax2c.set_visible(False)

if num_planewaves > 3:
    plane_tgb_br = array_to_scaled_rgba(scaled_beams[np.maximum(-num_planewaves+3,-1)],vmin=0,vmax=1)
    im_plane_br = ax2d.imshow(plane_tgb_br)
else:
    im_plane_br = ax2d.imshow(np.ones_like(plane_tgb_tl))
    ax2d.set_visible(False)
        
prism_fig.text(
    0.5,
    0.95,
    "largest-4 frequency beams",
    horizontalalignment="center",
    fontsize=12
)

probe_rgb = array_to_scaled_rgba(np.fft.fftshift(probe),vmin=0,vmax=1)
im_probe = ax3.imshow(probe_rgb)
ax3.set_title("Real-space converged probe")

for ax in prism_fig.axes:
    ax.set(xticks=[],yticks=[])

prism_fig.canvas.resizable = False
prism_fig.canvas.header_visible = False
prism_fig.canvas.footer_visible = False
prism_fig.canvas.toolbar_visible = False
prism_fig.canvas.layout.width = f'{width}px'
prism_fig.canvas.toolbar_position = 'bottom'
spec.tight_layout(prism_fig)
prism_fig.patch.set_alpha(0)
None

In [13]:
layout = ipywidgets.Layout(width=f'{width//2}px',height='30px')
style = {
    'description_width': 'initial',
}

In [14]:
def update_num_planewaves(change):
    num_planewaves = change["new"]
    
    inds, inds_i, inds_j = return_selected_inds(
        num_planewaves,
        arrays_mutated[1],
        arrays_mutated[2],
        arrays_mutated[3]
    )
    
    beams, scaled_beams, probe = return_arrays(
        arrays_mutated[0],
        inds,inds_i,inds_j,
        arrays_mutated[5],
        arrays_mutated[7]
    )
    
    beams_rgb = array_to_scaled_rgba(np.fft.fftshift(beams),vmin=0,vmax=1)
    im_beams.set_data(beams_rgb)

    plane_tgb_tl = array_to_scaled_rgba(scaled_beams[np.maximum(-num_planewaves,-4)],vmin=0,vmax=1)
    im_plane_tl.set_data(plane_tgb_tl)

    if num_planewaves > 1:
        plane_tgb_tr = array_to_scaled_rgba(scaled_beams[np.maximum(-num_planewaves+1,-3)],vmin=0,vmax=1)
        im_plane_tr.set_data(plane_tgb_tr)
        ax2b.set_visible(True)
    else:
        ax2b.set_visible(False)
    
    if num_planewaves > 2:
        plane_tgb_bl = array_to_scaled_rgba(scaled_beams[np.maximum(-num_planewaves+2,-2)],vmin=0,vmax=1)
        im_plane_bl.set_data(plane_tgb_bl)
        ax2c.set_visible(True)
    else:
        ax2c.set_visible(False)
    
    if num_planewaves > 3:
        plane_tgb_br = array_to_scaled_rgba(scaled_beams[np.maximum(-num_planewaves+3,-1)],vmin=0,vmax=1)
        im_plane_br.set_data(plane_tgb_br)
        ax2d.set_visible(True)
    else:
        ax2d.set_visible(False)


    probe_rgb = array_to_scaled_rgba(np.fft.fftshift(probe),vmin=0,vmax=1)
    im_probe.set_data(probe_rgb)
    
    prism_fig.canvas.draw_idle()
    return None

play = ipywidgets.Play(
    value=5,
    min=1,
    max=len(arrays_mutated[0]),
    step=1,
    interval=25,
)

slider = ipywidgets.IntSlider(
    min=1,
    max=len(arrays_mutated[0]),
    step=1,
    layout=layout,
    style=style,
    description="# of beams"
)

ipywidgets.jslink((play, 'value'), (slider, 'value'))
slider.observe(update_num_planewaves,"value")

In [15]:
def update_interpolation_factor(change):
    interpolation_factor = change["new"]
    
    s_matrix = abtem.SMatrix(
        potential=None,
        sampling=sampling,
        gpts=gpts,
        energy=energy,
        semiangle_cutoff=semiangle_cutoff,
        downsample=None,
        interpolation=interpolation_factor,
    ).build(
    )
    
    arrays_mutated[0] = s_matrix
    arrays_mutated[1:4] = return_ordering_indices(arrays_mutated[0])
    arrays_mutated[5] = return_position_coefs(arrays_mutated[0],arrays_mutated[4])
    arrays_mutated[7] = return_ctf_coefs(arrays_mutated[0],arrays_mutated[6])
    
    slider.max = len(arrays_mutated[0])
    if slider.value < slider.max:
        update_num_planewaves({"new":slider.value})
    return None

def update_defocus(change):

    arrays_mutated[6] = change["new"]
    arrays_mutated[7] = return_ctf_coefs(arrays_mutated[0],arrays_mutated[6])
    update_num_planewaves({"new":slider.value})
  
    return None

interpolation_slider = ipywidgets.SelectionSlider(
    options=[1,2,4,8,16],
    value=1,
    description="interpolation",
    layout=layout,
    style=style,
)
interpolation_slider.observe(update_interpolation_factor,"value")

defocus_slider = ipywidgets.FloatSlider(
    min=-150,
    max=150,
    value=0,
    description="defocus [Å]",
    layout=layout,
    style=style,
)
defocus_slider.observe(update_defocus,"value")

def onclick(event):
    """ """
    positions_px = np.array([event.ydata,event.xdata])
    
    if positions_px[0] is not None:
        arrays_mutated[4] = positions_px
        arrays_mutated[5] = return_position_coefs(arrays_mutated[0],arrays_mutated[4])
        update_num_planewaves({"new":slider.value})

cid = prism_fig.canvas.mpl_connect('button_press_event',onclick)

In [16]:
#| label: app:planewave_expansion_widget
ipywidgets.VBox(
    [
        ipywidgets.HBox([slider,play]),
        ipywidgets.HBox([interpolation_slider,defocus_slider]),
        prism_fig.canvas
    ],
    layout=ipywidgets.Layout(
        align_items="center"
    )
)